# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 protein dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset describes protein abundance, modifications, and sequences in human (Homo sapiens) samples as mass spectrometry analysis results.

- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- License: [ODC-BY-1.0](https://opendatacommons.org/licenses/by/1-0/)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata object
dataset = mlc.Dataset(croissant_url)

# Show metadata overview
print('Dataset Name:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Version:', dataset.metadata.version)
print('Published on:', dataset.metadata.datePublished)
print('Keywords:', dataset.metadata.keywords)
print('License:', dataset.metadata.license)

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by the schema.

Each data entity is referenced by its `@id`. Use these IDs to extract and analyze data.

In [ ]:
# List available record sets by their `@id` and main fields
record_sets = list(dataset.record_sets())
print('Available record sets in schema:')
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    print('  Fields:')
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    - Field @id: {field.get('@id')}, name: {field.get('name', '')}")
        else:
            print(f"    - Field @id: {field}")

Below, we show an example of the first record from the main protein record set. We use the `@id` for loading.

In [ ]:
# Preview a record from one record set (using its @id).
# (Change the @id below to one listed in the previous cell for specific data.)
main_record_set_id = None
# Try to auto detect main record set id (the one that contains protein/peptide info)
for rs in record_sets:
    n = str(rs.get('name', '')).lower()
    if 'protein' in n or 'main' in n:
        main_record_set_id = rs['@id']
if not main_record_set_id:
    main_record_set_id = record_sets[0]['@id']  # default to first

print(f"Using record set @id: {main_record_set_id}")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i == 0:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

You should reference record sets and fields by their `@id`s for consistency.

In [ ]:
# Extract data from all record sets as individual DataFrames
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set @id: {rs_id} with {len(df)} rows and {len(df.columns)} columns.")
    else:
        print(f"No records found for record set @id: {rs_id}.")

# Show columns of the main protein record set
print('\nMain record set columns:')
main_df = dataframes[main_record_set_id]
print(main_df.columns.tolist())

# Show sample of data
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering by numeric field value, normalizing, and grouping using field `@id`s.

Here we'll attempt to analyze a numeric field (e.g., `MW [kDa]`, `Coverage [%]`, or abundance field) if present.

In [ ]:
# Try to auto-detect a numeric field for EDA (adjust @id to match actual data)
numeric_field_id = None
for col in main_df.columns:
    # Common numeric field names
    if any(s in col.lower() for s in ['mw', 'weight', 'abund', 'coverage', 'count', 'peptide', 'score']):
        # Exclude non-numeric or string-heavy columns
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
        # Try convert to numeric
        try:
            converted = pd.to_numeric(main_df[col], errors='coerce')
            if converted.notna().sum() > 0.5 * len(converted):
                main_df[col] = converted
                numeric_field_id = col
                break
        except Exception:
            continue
if not numeric_field_id:
    # Just pick the first numeric column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

print(f"Selected numeric field for EDA: {numeric_field_id}")

threshold = main_df[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy() if numeric_field_id else main_df.copy()
print(f"Filtered records: {len(filtered_df)} have {numeric_field_id} > {threshold:.2f}")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field (e.g. description, accession, or source)
group_field_candidates = [c for c in main_df.columns if any(s in c.lower() for s in ["desc", "access", "sample", "type", "id"])]
group_field = None
for col in group_field_candidates:
    if main_df[col].nunique() < 20 and main_df[col].nunique() > 1:
        group_field = col
        break
if group_field is None and len(main_df.columns) > 1:
    group_field = main_df.columns[0]

print(f"Attempting to group by: {group_field}")
if group_field in filtered_df.columns and group_field != numeric_field_id:
    grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of filtering/normalizing.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(10,4))
    main_df[numeric_field_id].hist(bins=30, alpha=0.6, label='All records')
    filtered_df[numeric_field_id].hist(bins=30, alpha=0.6, label='Filtered')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.legend()
    plt.show()
    
    # If normalized field exists, show
    if f"{numeric_field_id}_normalized" in filtered_df:
        plt.figure(figsize=(8,4))
        filtered_df[f"{numeric_field_id}_normalized"].plot.hist(bins=20, title=f"Normalized {numeric_field_id} (filtered)")
        plt.show()

## 6. Conclusion

This notebook demonstrated the following:

- Loading the protein dataset and schema with `mlcroissant`.
- Identifying record sets and using their `@id`.
- Extracting data and accessing fields by `@id`.
- Filtering and normalizing records for numeric fields.
- Visualizing the distribution of quantitative variables.

For full scientific usage, please consult column and field documentation in the FAIR^2 package.